# 企业级多智能体协同系统

电商客服订单处理案例，使用 AutoGen 框架实现多任务协同，包括客服流程拆解、数据查询联动和跨部门协作调度。

适用场景：电商客服系统订单问题处理。

## 0. 环境准备

运行前请确认已经安装 AutoGen 相关依赖，并在环境变量中配置 OPENAI_API_KEY 与 OPENAI_API_BASE。

如果你刚更新了 week01 的依赖，请先在 week01 目录执行 uv sync，再按顺序运行下面的单元。

In [22]:
import asyncio
import json
import os

from dotenv import load_dotenv
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_API_BASE")

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=api_key,
    base_url=base_url,
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "family": "gpt",
        "structured_output": True,
    },
)
print(base_url)

https://api.vveai.com/v1


## 1. 构建模拟企业数据服务

先定义一个内存中的企业数据层，用它模拟订单、库存和物流系统。

In [23]:
from typing import Dict


class EnterpriseDataService:
    """企业数据服务模拟类"""

    def __init__(self):
        self.orders = {
            "ORD001": {
                "order_id": "ORD001",
                "customer_id": "CUST001",
                "status": "已发货",
                "items": [{"product": "iPhone 15", "quantity": 1, "price": 7999}],
                "total": 7999,
                "shipping_address": "北京市朝阳区xxx街道",
                "tracking_number": "SF1234567890",
                "order_date": "2024-01-15",
                "expected_delivery": "2024-01-18",
            },
            "ORD002": {
                "order_id": "ORD002",
                "customer_id": "CUST002",
                "status": "处理中",
                "items": [{"product": "MacBook Pro", "quantity": 1, "price": 15999}],
                "total": 15999,
                "shipping_address": "上海市浦东新区xxx路",
                "tracking_number": None,
                "order_date": "2024-01-16",
                "expected_delivery": None,
            },
        }

        self.inventory = {
            "iPhone 15": {"stock": 100, "warehouse": "华北仓"},
            "MacBook Pro": {"stock": 0, "warehouse": "华东仓"},
        }

        self.logistics = {
            "SF1234567890": {
                "status": "运输中",
                "current_location": "北京分拣中心",
                "estimated_arrival": "2024-01-18 14:00",
            }
        }

    def get_order_info(self, order_id: str) -> Dict:
        return self.orders.get(order_id, {})

    def get_inventory_info(self, product: str) -> Dict:
        return self.inventory.get(product, {})

    def get_logistics_info(self, tracking_number: str) -> Dict:
        return self.logistics.get(tracking_number, {})

    def update_order_status(self, order_id: str, new_status: str) -> bool:
        if order_id in self.orders:
            self.orders[order_id]["status"] = new_status
            return True
        return False

## 2. 封装供智能体调用的工具

这一层把企业数据服务包装成异步工具函数，后续可以直接挂到不同的智能体上。

In [24]:
data_service = EnterpriseDataService()


async def get_order_info(order_id: str) -> str:
    try:
        order_info = data_service.get_order_info(order_id)
        if order_info:
            return f"订单信息查询成功：\n{json.dumps(order_info, ensure_ascii=False, indent=2)}"
        return f"未找到订单号 {order_id} 的信息，请检查订单号是否正确。"
    except Exception as exc:
        return f"查询订单信息时出错：{str(exc)}"


async def get_inventory_info(product: str) -> str:
    try:
        inventory_info = data_service.get_inventory_info(product)
        if inventory_info:
            return f"库存信息查询成功：\n{json.dumps(inventory_info, ensure_ascii=False, indent=2)}"
        return f"未找到产品 {product} 的库存信息。"
    except Exception as exc:
        return f"查询库存信息时出错：{str(exc)}"


async def get_logistics_info(tracking_number: str) -> str:
    try:
        logistics_info = data_service.get_logistics_info(tracking_number)
        if logistics_info:
            return f"物流信息查询成功：\n{json.dumps(logistics_info, ensure_ascii=False, indent=2)}"
        return f"未找到运单号 {tracking_number} 的物流信息。"
    except Exception as exc:
        return f"查询物流信息时出错：{str(exc)}"

## 3. 定义智能体角色

下面开始逐步定义客服接待员、专业查询专员、用户代理以及群组聊天的调度逻辑。

In [25]:
customer_service_agent = AssistantAgent(
    name="customer_service_agent",
    model_client=model_client,
    system_message="""你是一名专业的电商客服接待员。你的职责是：
1. 友好接待客户，了解客户问题
2. 对问题进行初步分类（订单查询、退换货、物流问题、产品咨询等）
3. 收集必要的订单信息（订单号、客户信息等）
4. 将问题转交给相应的专业团队处理

请用简洁明了的语言与客户沟通。当客户提到具体订单号时，请直接转交给订单查询专员处理。
如果问题涉及多个方面，请协调相关专员共同解决。

回复格式：简洁专业，直接回答客户问题。""",
    reflect_on_tool_use=True,
    model_client_stream=True,
)

### 3.1 订单、物流与库存专员

客服接待员只负责识别问题，真正的数据查询由三个专业代理分别完成。

In [26]:
order_query_agent = AssistantAgent(
    name="order_query_agent",
    model_client=model_client,
    tools=[get_order_info],
    system_message="""你是订单查询专员，负责处理所有订单相关的查询。你的职责包括：
1. 根据订单号查询订单详细信息
2. 解释订单状态和处理进度
3. 提供预计发货和到货时间
4. 识别需要其他部门协助的问题

当客户提供订单号时，请立即使用 get_order_info 函数查询订单信息。
根据查询结果，如果发现需要物流或库存部门协助，请主动通知相关专员。

回复格式：提供详细的订单信息，包括状态、商品、金额等关键信息。""",
    reflect_on_tool_use=True,
    model_client_stream=True,
)

logistics_agent = AssistantAgent(
    name="logistics_agent",
    model_client=model_client,
    tools=[get_logistics_info],
    system_message="""你是物流跟踪专员，专门处理配送和物流相关问题。你的职责包括：
1. 查询包裹物流状态和位置
2. 提供准确的配送时间预估
3. 处理配送异常和延误问题
4. 协调配送地址修改

当需要查询物流信息时，请使用 get_logistics_info 函数。
请提供实时、准确的物流信息，并主动提醒客户注意事项。

回复格式：提供详细的物流状态，包括当前位置、预计到达时间等。""",
    reflect_on_tool_use=True,
    model_client_stream=True,
)

inventory_agent = AssistantAgent(
    name="inventory_agent",
    model_client=model_client,
    tools=[get_inventory_info],
    system_message="""你是库存管理专员，负责处理库存相关问题。你的职责包括：
1. 查询产品库存状态
2. 预估补货时间
3. 协调缺货订单处理
4. 提供替代产品建议

当需要查询库存信息时，请使用 get_inventory_info 函数。
请提供准确的库存信息，并为缺货情况提供合理的解决方案。

回复格式：提供库存状态，如果缺货请说明预计补货时间。""",
    reflect_on_tool_use=True,
    model_client_stream=True,
)

### 3.2 用户代理与群组聊天调度

这一部分补上模拟用户、终止条件和 SelectorGroupChat，让多个智能体能够自动轮流发言。

In [27]:
def auto_reply_input(input_prompt: str = "") -> str:
    _ = input_prompt
    return "谢谢您的帮助，问题已解决!"


user_agent = UserProxyAgent(
    name="user_agent",
    description="模拟用户行为，用于自然结束对话",
    input_func=auto_reply_input,
 )


def create_group_chat():
    text_termination = (
        TextMentionTermination("谢谢您的帮助")
        | TextMentionTermination("问题已解决")
        | TextMentionTermination("已解决")
    )
    max_msg_termination = MaxMessageTermination(max_messages=12)
    termination_condition = text_termination | max_msg_termination

    return SelectorGroupChat(
        [customer_service_agent, order_query_agent, logistics_agent, inventory_agent, user_agent],
        model_client=model_client,
        termination_condition=termination_condition,
        selector_prompt="""
你正在进行一个角色扮演游戏。可用的角色如下：
{roles}
请阅读以下对话内容，然后从{participants}中选择下一个要发言的角色。只需返回角色名称。
{history}
请阅读上述对话，然后从{participants}中选择下一个要发言的角色。只需返回角色名称。
""",
    )

## 4. 运行单个客服场景

先封装一个辅助函数，用于展示标题、创建群组聊天并执行一次完整对话。

In [28]:
async def run_scenario_with_autogen(scenario_name: str, customer_message: str):
    print(f"\n{'=' * 60}")
    print(f"🎯 {scenario_name}")
    print(f"{'=' * 60}")
    print(f"客户问题：{customer_message}")
    print("\n🤖 AutoGen 多智能体协作处理：")
    print("-" * 50)

    group_chat = create_group_chat()
    await Console(group_chat.run_stream(task=customer_message))
    print("\n✅ 场景处理完成")

## 5. 批量运行多个客服场景

最后把单个场景执行函数组合起来，按顺序演示多个客服问题的完整协同过程。

In [29]:
async def main():
    print("🏢 企业级多智能体协同系统 - 电商客服订单处理演示")
    print("基于 AutoGen 框架实现")
    print("=" * 80)
    print("系统特性：")
    print("✅ 1. 客服流程自动拆解")
    print("✅ 2. 多数据源联动查询")
    print("✅ 3. 跨部门智能协作")
    print("✅ 4. 问题升级和路由")
    print("✅ 5. AutoGen 框架支持")

    scenarios = [
        ("场景1：订单状态查询", "你好，我想查询一下我的订单ORD001的状态，什么时候能到货？"),
        ("场景2：缺货问题处理", "我下单的MacBook Pro订单ORD002一直显示处理中，什么时候能发货？"),
        ("场景3：物流延误处理", "我的订单ORD001已经超过预计到货时间了，但还没收到货，这是怎么回事？"),
    ]

    try:
        for scenario_name, scenario_message in scenarios:
            await run_scenario_with_autogen(scenario_name, scenario_message)
            await asyncio.sleep(2)

        print(f"\n{'=' * 80}")
        print("🎉 企业级多智能体协同演示完成！")
        print("💡 该系统基于 AutoGen 框架，展示了电商客服系统中的多任务协同和跨部门协作")
    finally:
        await model_client.close()

## 6. 执行演示

执行下面的单元会按顺序运行 3 个客服场景。由于 main 会关闭模型客户端，若要重新运行，请先重新执行前面的代码单元。

In [30]:
await main()

🏢 企业级多智能体协同系统 - 电商客服订单处理演示
基于 AutoGen 框架实现
系统特性：
✅ 1. 客服流程自动拆解
✅ 2. 多数据源联动查询
✅ 3. 跨部门智能协作
✅ 4. 问题升级和路由
✅ 5. AutoGen 框架支持

🎯 场景1：订单状态查询
客户问题：你好，我想查询一下我的订单ORD001的状态，什么时候能到货？

🤖 AutoGen 多智能体协作处理：
--------------------------------------------------
---------- TextMessage (user) ----------
你好，我想查询一下我的订单ORD001的状态，什么时候能到货？


/Users/snow/Documents/Repository/ai-engineer-training/.venv/lib/python3.12/site-packages/autogen_agentchat/teams/_group_chat/_selector_group_chat.py:269: UserWarning: Resolved model mismatch: gpt-4o-mini-2024-07-18 != gpt-4o-mini. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-mini to enhance token/cost estimation and suppress this warning.
  response = await self._model_client.create(messages=select_speaker_messages)


---------- ToolCallRequestEvent (order_query_agent) ----------
[FunctionCall(id='call_OAikCPYxWLfXGHJPS0nNfO0B', arguments='{"order_id":"ORD001"}', name='get_order_info')]
---------- ToolCallExecutionEvent (order_query_agent) ----------
[FunctionExecutionResult(content='订单信息查询成功：\n{\n  "order_id": "ORD001",\n  "customer_id": "CUST001",\n  "status": "已发货",\n  "items": [\n    {\n      "product": "iPhone 15",\n      "quantity": 1,\n      "price": 7999\n    }\n  ],\n  "total": 7999,\n  "shipping_address": "北京市朝阳区xxx街道",\n  "tracking_number": "SF1234567890",\n  "order_date": "2024-01-15",\n  "expected_delivery": "2024-01-18"\n}', name='get_order_info', call_id='call_OAikCPYxWLfXGHJPS0nNfO0B', is_error=False)]
---------- ModelClientStreamingChunkEvent (order_query_agent) ----------
您的订单信息如下：

- **订单号**: ORD001
- **客户ID**: CUST001
- **订单状态**: 已发货
- **商品**: iPhone 15
- **数量**: 1
- **单价**: 7999元
- **总金额**: 7999元
- **收货地址**: 北京市朝阳区xxx街道
- **快递单号**: SF1234567890
- **订单日期**: 2024-01-15
- **预计到货时间*